# ENVIRA Gradio PDF Layout Application

Run all cells. This notebook only bootstraps the repository and environment,
mounts Google Drive, initializes the standalone application, and launches Gradio.


## 1. Runtime / repository setup

Reuse a valid checkout. Clone only when missing; try the public URL before requesting credentials.


In [ ]:
from getpass import getpass
from pathlib import Path
import os, shutil, stat, subprocess, tempfile

WORKSPACE_DIR = Path("/content/colab_repos")
REPO_DIR = WORKSPACE_DIR / "ENVIRA-Phase-1-Data"
APP_DIR = REPO_DIR / "envira_gradio_web_app"
REPOSITORY_URL = "https://github.com/dsdengrasa08/ENVIRA-Phase-1-Data.git"
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

def run_clone(env=None):
    return subprocess.run(
        ["git", "clone", "--depth", "1", REPOSITORY_URL, str(REPO_DIR)],
        text=True, capture_output=True, env=env, check=False,
    )

if REPO_DIR.is_dir() and (REPO_DIR / ".git").is_dir():
    print(f"Reusing repository: {REPO_DIR}")
    # A Colab runtime can outlive an application update. Fast-forward a clean
    # checkout so Run All does not silently reinstall an obsolete wheel.
    clean = not subprocess.run(
        ["git", "-C", str(REPO_DIR), "status", "--porcelain"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    if clean:
        update = subprocess.run(
            ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
            capture_output=True, text=True, check=False,
            env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
        )
        if update.returncode:
            print("Warning: existing checkout could not be updated; using its current commit.")
        else:
            print("Existing checkout updated with a fast-forward pull.")
    else:
        print("Existing checkout has local changes; skipping automatic update.")
else:
    if REPO_DIR.exists():
        raise RuntimeError(f"Refusing to overwrite non-Git path: {REPO_DIR}")
    result = run_clone({**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    if result.returncode:
        token = getpass("GitHub token (requested only because public clone failed): ")
        with tempfile.TemporaryDirectory() as helper_dir:
            helper = Path(helper_dir) / "askpass.sh"
            helper.write_text('#!/bin/sh\ncase "$1" in *Username*) echo "$GITHUB_USER";; *) echo "$GITHUB_TOKEN";; esac\n')
            helper.chmod(stat.S_IRUSR | stat.S_IWUSR | stat.S_IXUSR)
            env = {**os.environ, "GIT_ASKPASS": str(helper), "GIT_TERMINAL_PROMPT": "0",
                   "GITHUB_USER": "x-access-token", "GITHUB_TOKEN": token}
            result = run_clone(env)
        del token
    if result.returncode:
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        raise RuntimeError("Repository clone failed; credentials and clone output were not printed.")

if not (REPO_DIR / ".git").is_dir() or not (APP_DIR / "pyproject.toml").is_file():
    raise RuntimeError("Checkout does not contain the standalone Gradio application.")
print(subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())


## 2. Environment setup

Install only the independent application package.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(APP_DIR)], check=True)


## 3. Google Drive

Mount Drive once and configure the authoritative persistent root.


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
PERSISTENT_ROOT = Path("/content/drive/MyDrive/ENVIRA/pdf_layout_gradio")
PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Persistent output root: {PERSISTENT_ROOT}")


## 4. Application initialization

Load configuration, validate persistent models, and initialize the reusable Docling converter once.


In [ ]:
from envira_gradio import (
    close_application, create_app, initialize_application, launch_application,
)
from envira_gradio.settings import AppSettings

settings = AppSettings(
    persistent_root=PERSISTENT_ROOT,
    config_path=APP_DIR / "config" / "colab.yaml",
    temporary_root=Path("/content/envira_gradio_tmp"),
    model_root=PERSISTENT_ROOT / "models" / "docling",
)
runtime = initialize_application(settings)
demo = create_app(runtime)
print("Application initialized.")


## 5. Gradio launch


In [ ]:
launch_info = launch_application(
    demo,
    share=True,
    height=900,
    max_share_attempts=2,
    retry_delay_seconds=2.0,
)
print(f"Presentation mode: {launch_info.presentation}")
print(f"Local server: {launch_info.local_url}")
print(f"Native share attempts: {launch_info.share_attempts}")
if launch_info.share_url:
    print(f"Public share URL: {launch_info.share_url}")
else:
    print(f"Public sharing unavailable: {launch_info.share_failure}")
    print("Using the authenticated Colab kernel proxy; PDF processing is unaffected.")
    print("Share diagnostics:", launch_info.diagnostics)


## 6. Server lifecycle

Closing the browser tab or embedded app does not close the Colab runtime. To stop only the Gradio server, run the next command manually. Rerunning the launch cell safely closes an existing Gradio server first. A Colab runtime disconnect/restart stops the server and clears in-memory models, but persistent Google Drive outputs remain.


In [ ]:
# Run manually only when you want to stop the web server:
# close_application(demo)
